# Training notebook - QLoRA fine-tune of the Bengali hallucination judge
## A **negative result**. Nothing in this notebook ships.

> **Read this first.** Our submitted system uses **zero fine-tuning**. The judge is the stock, public
> `Qwen/Qwen3-32B-AWQ`, unmodified. This notebook documents a fine-tuning track we ran to completion,
> measured, and **rejected** - it is included because the measurement is a result in its own right,
> and because "we chose not to fine-tune" is a much weaker claim than "we fine-tuned and it lost."

### The headline

| Fine-tune | Train rows | Base (dev macro-F1) | Tuned | Delta |
|---|---:|---:|---:|---:|
| **Qwen3-32B**, QLoRA r=16, 1 epoch | 60,026 | **0.832** | **0.708** | **-0.124** |
| **TigerLLM-9B**, QLoRA | 20,000 | 0.748 | 0.646 | -0.102 |

The collapse concentrated in **exactly the band it was built to fix** - closed-book cultural knowledge
went **0.809 -> 0.618 (-0.191)** - and *every* domain with a dedicated data slice got worse
(idiom error 20% -> 50%, math error 0% -> 44%).

For comparison, the shipped retrieval + routing system scores **0.913**. On this task and budget,
**corpus curation beat weight updates by +0.205 macro-F1**.

### Why we tried it at all

At the time, retrieval, ensembling and routing all looked tapped at ~0.853-0.860, while a frontier
proprietary model scored 0.912 on the same dev set. That gap read as a *knowledge* gap, not a reasoning
gap - and internalising Bengali cultural knowledge into the weights was the one untried lever. It was a
reasonable bet. It lost, and then a further corpus wave took the frozen judge to 0.913 instead.

### Runnability

This is a **lab-machine** notebook, not a Kaggle one (the rules make the training notebook optional and
do not require it to be Kaggle-runnable). It needs a single **24 GB+ bf16 GPU** (3090/4090/A5000/L4) and
the `FineTuning/` bundle for its data and render module. The source of truth is
[`FineTuning/train_qlora.py`](https://github.com/farhanishraq17/-CutiePookieUrza---Bengali-LLM-Hallucination-Detection);
the code cells below are that script, split for reading.

## 1 - Setup

Unsloth 4-bit QLoRA: ~18 GB for the base + ~2-4 GB for LoRA/optimizer/activations. `--no-deps` on
unsloth mirrors the pattern used in the inference kernel - it must not disturb the pinned torch.

In [ ]:
# pip install -q --no-deps unsloth unsloth_zoo
# pip install -q trl peft accelerate bitsandbytes datasets

import os, io, json, re

## 2 - Configuration

`enable_thinking=False` throughout: Qwen3's thinking toggle is turned **off** for training and inference
so the model answers straight to `FINAL: 0|1`. That was a deliberate budget choice - a thinking judge at
5-6 s/row was the thing we were trying to make cheaper.

**In hindsight this was already a warning sign.** On the frozen judge, turning reasoning *off* costs
**-0.075** macro-F1: the reasoning tokens do real verification work. We trained the model into the very
configuration we had measured to be worse.

In [ ]:
SMOKE      = os.environ.get("SMOKE", "0") == "1"
# Pre-quantized 4-bit Qwen3 -> ~20 GB download instead of 65 GB fp16 (matters on a 24 GB card).
BASE       = os.environ.get("BASE", ("unsloth/Qwen2.5-14B-Instruct-bnb-4bit" if SMOKE
                                     else "unsloth/Qwen3-32B-bnb-4bit"))
TRAIN      = os.environ.get("TRAIN", "ftgen/data_ft/train_qwen3.jsonl")   # 60,026 rows
OUT        = os.environ.get("OUT", "out_ft")
MAX_SEQ    = int(os.environ.get("MAX_SEQ", "2048"))
N_TRAIN    = int(os.environ.get("N_TRAIN", "4000" if SMOKE else "0"))     # 0 = all
EPOCHS     = float(os.environ.get("EPOCHS", "1"))
MAX_STEPS  = int(os.environ.get("MAX_STEPS", "60" if SMOKE else "0"))     # 0 = use EPOCHS
LORA_R     = int(os.environ.get("LORA_R", "16"))
BATCH      = int(os.environ.get("BATCH", "1"))
ACCUM      = int(os.environ.get("ACCUM", "8"))
EVAL_299   = "ds_ft/eval_299.json"    # the pre-registered held-out gate
EVAL_724   = "ds_ft/eval_724.json"

## 3 - The training data (60,026 rows)

Balanced 50/50 faithful/hallucinated, assembled from bnMMLU, idioms/bagdhara, math/somadhan, GK, BNLI,
TyDiQA, BD-wiki, idiom hard-negatives and grammar - i.e. a dedicated slice for **every** domain where the
frozen judge was known to fail. That is what makes the outcome informative: the failure is not "we
targeted the wrong thing."

Each row is `{messages: [system, user], target: "...\nFINAL: 0|1"}`, rendered by `ftgen/render.py` with
the **byte-identical** SYS/user templates used at inference, so train and inference formats match.

In [ ]:
def load_jsonl(fp, limit=0):
    txt = io.open(fp, encoding="utf-8").read().lstrip()
    if txt[:1] == "[":
        rows = json.loads(txt)                                          # JSON array (eval sets)
    else:
        rows = [json.loads(l) for l in txt.splitlines() if l.strip()]   # JSONL (train set)
    return rows[:limit] if limit else rows

from unsloth import FastLanguageModel
import torch
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

model, tok = FastLanguageModel.from_pretrained(
    model_name=BASE, max_seq_length=MAX_SEQ, load_in_4bit=True, dtype=None)

## 4 - Zero-shot baseline **before** training

Pre-registered gate: measure the untuned model on the held-out sets *first*, and never submit a tuned
model that does not beat it. This cell is the reason the failure cost us nothing but time.

In [ ]:
_FINAL = re.compile(r"FINAL:\s*([01])")
def _parse(text):
    m = list(_FINAL.finditer(text))
    if m:
        return int(m[-1].group(1))
    low = text.lower()
    return 0 if ("hallucinat" in low or "not faithful" in low) else 1   # default faithful

from ftgen.render import SYS, _user     # byte-identical to the inference prompts

def predict(model, tok, rows, tag="eval", gen_tokens=512):
    FastLanguageModel.for_inference(model)
    preds = {}
    for i, r in enumerate(rows):
        ctx = r.get("context") or "[NULL]"
        msgs = [{"role": "system", "content": SYS},
                {"role": "user", "content": _user(ctx, r["prompt_bn"], r["response_bn"])}]
        prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                         enable_thinking=False)
        ids = tok(prompt, return_tensors="pt").to(model.device)
        out = model.generate(**ids, max_new_tokens=gen_tokens, do_sample=False,
                             temperature=None, top_p=None, pad_token_id=tok.eos_token_id)
        preds[str(r.get("id", i))] = _parse(tok.decode(out[0][ids.input_ids.shape[1]:],
                                                       skip_special_tokens=True))
    return preds

EVAL_N = int(os.environ.get("EVAL_N", "120"))
e299, e724 = load_jsonl(EVAL_299, EVAL_N), load_jsonl(EVAL_724, EVAL_N)

os.makedirs(OUT, exist_ok=True)
json.dump(predict(model, tok, e299, "zs299", 1024), io.open(f"{OUT}/zs_299.json", "w"))
json.dump(predict(model, tok, e724, "zs724", 1024), io.open(f"{OUT}/zs_724.json", "w"))

## 5 - LoRA + trainer config

### FLAW #1 - packing without response masking (the fatal one)

`packing="bfd"` bins examples into 2048-token blocks. It is fast and wastes no padding, but with packing
on, TRL cannot apply per-example prompt masking - so the loss is computed over the **entire packed
block**: system prompt, question, context, response *and* verdict alike.

The verdict is roughly **10 tokens out of 2048**. We therefore spent ~99.5% of the gradient budget
teaching the model to reproduce Bengali question text it already knew, and ~0.5% teaching it the one
thing we wanted - the judgement. The `train_on_responses_only` path that fixes this **exists in the code
below but is unreachable while `packing=True`**, which is precisely the bug.

### FLAW #2 - one full epoch, `save_strategy="no"`

60,026 rows at accum 8 is ~7,500 optimizer steps in a single pass, with **no intermediate checkpoints
saved**. When the final model came out worse there was no earlier checkpoint to fall back to, so we could
not even find out whether an earlier stopping point would have helped. The run was unsalvageable by
construction.

### FLAW #3 - LR 1e-4 for r=16

Aggressive for this rank on a 32B base. Combined with #1 and #2, the model drifted away from its
calibrated prior and started over-flagging - which is the specific way it lost: a judge that was
well-calibrated toward "faithful when unsure" became trigger-happy.

In [ ]:
def to_text(ex):
    msgs = ex["messages"] + [{"role": "assistant", "content": ex["target"]}]
    return {"text": tok.apply_chat_template(msgs, tokenize=False, enable_thinking=False)}

ds = Dataset.from_list(load_jsonl(TRAIN, N_TRAIN)).map(to_text, remove_columns=[])
print(f"train rows: {len(ds)}")

model = FastLanguageModel.get_peft_model(
    model, r=LORA_R, lora_alpha=LORA_R, lora_dropout=0.0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=13)

cfg = SFTConfig(
    output_dir=f"{OUT}/ckpt", per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=ACCUM, warmup_ratio=0.03,
    num_train_epochs=EPOCHS, max_steps=(MAX_STEPS or -1),
    learning_rate=1e-4, logging_steps=10,
    save_strategy="no",              # <-- FLAW #2: no early checkpoint to fall back to
    optim="adamw_8bit", weight_decay=0.01, lr_scheduler_type="cosine",
    seed=13, max_seq_length=MAX_SEQ, dataset_text_field="text", report_to="none",
    packing=True, packing_strategy="bfd")   # <-- FLAW #1: forces training on full text

trainer = SFTTrainer(model=model, tokenizer=tok, train_dataset=ds, args=cfg)

# The masking path below is CORRECT but unreachable while packing=True. Fixing this run means
# packing=False + this block enabled, so the loss lands on the assistant completion only.
if not getattr(cfg, "packing", False):
    from unsloth.chat_templates import train_on_responses_only
    trainer = train_on_responses_only(
        trainer, instruction_part="<|im_start|>user\n",
        response_part="<|im_start|>assistant\n")
    print("train_on_responses_only: ON")
else:
    print("packing=True -> training on full text (prompt-masking skipped)")

trainer.train()          # ~24 h on an RTX 3090

## 6 - Evaluate the tuned model against the gate

In [ ]:
json.dump(predict(model, tok, e299, "ft299", 512), io.open(f"{OUT}/ft_299.json", "w"))
json.dump(predict(model, tok, e724, "ft724", 512), io.open(f"{OUT}/ft_724.json", "w"))

model.save_pretrained(f"{OUT}/lora"); tok.save_pretrained(f"{OUT}/lora")
try:
    model.save_pretrained_merged(f"{OUT}/merged", tok, save_method="merged_16bit")
except Exception as e:
    print(f"[warn] merged save skipped: {e}")     # 4-bit -> fp16 merge also failed on this run

import eval_ft
eval_ft.report(EVAL_299, f"{OUT}/zs_299.json", f"{OUT}/ft_299.json", "ds_ft/all_categories.json")
eval_ft.report(EVAL_724, f"{OUT}/zs_724.json", f"{OUT}/ft_724.json", None)

## 7 - Result, and what we did about it

```
dev macro-F1     zero-shot 0.832  ->  fine-tuned 0.708   (-0.124)
closed-book band     0.809        ->      0.618          (-0.191)
idiom error rate      20%         ->       50%
math error rate        0%         ->       44%
```

Every band regressed, worst in the band the training data was built to fix. Two secondary failures
compounded it: the 4-bit -> fp16 merge failed, and checkpoint pruning removed any earlier state to
salvage.

**The gate did its job.** Because the zero-shot baseline was measured *before* training on the same
held-out rows, the regression was caught without spending a leaderboard submission on it. We dropped the
fine-tuning track and put the remaining time into corpus curation - which then delivered
**0.894 -> 0.913** on the real leaderboard.

### What we would do differently

1. **`packing=False` + `train_on_responses_only`** - put the loss on the ~10 verdict tokens, not on 2048
   tokens of Bengali the model already knows. This is the change we would make first.
2. **Checkpoint every ~500 steps and evaluate each** - 1 full epoch was almost certainly past the useful
   point, and we had no way to find out.
3. **LR 2e-5 - 5e-5 for r=16**, and keep thinking **on** given the measured -0.075 for turning it off.

We are careful about the scope of this claim: **fine-tuning is not refuted for this task** - our
*recipe* is. What the experiment does establish is that under a fixed budget, curating retrievable
knowledge was the higher-return use of the same GPU-hours. That is the finding, and it is the one the
paper reports.

### Related

- [`../MODELS.md`](../MODELS.md) - every model we evaluated, per-band results, and the ensembling study
- [`../docs/RESULTS.md`](../docs/RESULTS.md) - the system-level progression and lever ablations
- [`../notebook/CUTIEPOOKIEURZA_inference_notebook.ipynb`](../notebook/CUTIEPOOKIEURZA_inference_notebook.ipynb) - the system that actually ships